# West West Minya Center-Pivot Water Productivity Analysis
### Satellite-Based Irrigation Monitoring · West Minya Governorate, Egypt

**Author:** **Nour Negm**· PhD | Plant Genetics & Breeding | Remote Sensing & Data Analytics for Agriculture & Water | Egypt–MENA | Year: **2026**

---

> **Copyright & Usage Terms**

> This notebook and all its contents — including analytical workflows, figures, and derived
> outputs — are the intellectual property of the author. Licensed under
> [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/).

> Free for educational and research use with attribution.
> Commercial use requires explicit written permission from the author.

> Licensing inquiries: https://www.linkedin.com/in/nour-ibrahim/

---

**Monitoring Period:** Winter Seasons 2021/22 – 2024/25 (November – April)

**Analytical Framework:** FAO WaPOR v3 L3 · WaPOR PCP L1 · Sentinel-2 SAVI (AWS Element84 STAC) · Mann-Kendall Trend Analysis

---

This notebook implements a complete remote sensing pipeline to quantify land expansion,
groundwater demand, and biomass water productivity across **166 center-pivot irrigation
systems** in the West West Minya reclamation zone. The cluster draws groundwater from the Middle Eocene fractured limestone aquifer (well depths 420–750m). All analyses are based on open satellite
data and follow FAO WaPOR benchmarking methodology.

**Key References:**
## Key References

- Allen et al. (1998) — FAO Irrigation and Drainage Paper 56
- Alsayyad et al. (2024) — Geology and agricultural impact, West Minia
- FAO (2023) — WaPOR Database Methodology v3
- Huete (1988) — SAVI · *Remote Sensing of Environment*
- IHE Delft (2020) — Water Productivity and Water Accounting using WaPOR
- Khalil et al. (2024) — Eocene carbonate aquifer, West Al-Minya · *Water*
- Mann (1945) — *Econometrica* · Sen (1968) — *JASA*
- Morsy (2023) — Groundwater management, West-West Minya · *Applied Water Science*
- Shams et al. (2025) — Aquifer degradation 2016–2024, West Mallawi
- Steduto et al. (2012) — FAO Irrigation and Drainage Paper 66
- Zwart & Bastiaanssen (2004) — *Agricultural Water Management*

##Import Libraries

In [ ]:
#Install Dependencies
!pip install -q pymannkendall rasterstats rioxarray netCDF4 xarray --quiet

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

import xarray as xr
import rioxarray as rxr
import rasterstats
import pymannkendall as mk

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import seaborn as sns
from shapely.wkt import loads as wkt_loads

# Configure logging and warnings
warnings.filterwarnings('ignore', category=UserWarning, module='rioxarray')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='rasterstats')

# Standardized Plotting Aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titleweight': 'bold',
    'figure.dpi': 150,
    'savefig.bbox': 'tight',
})

# Project Constants
SEASON_LABELS = ['2021/22', '2022/23', '2023/24', '2024/25']
N_SEASONS = 4

# Hydrological & Technical Assumptions:
# Ref: Howell (2003) & FAO 24 (Modern Center-Pivot Benchmark)
APPLICATION_EFFICIENCY = 0.85

# Ref: FAO 56 & CROPWAT (Fixed Percentage Method for Arid Zones)
EFF_RAINFALL_COEFF = 0.80

print('Environment and project constants initialized.')

## 1 · Environment Configuration

Paths to WaPOR seasonal and dekadal cubes, SAVI classification output, and project output directories are configured here. All downstream cells depend on these variables.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root project directory
project_dir = r"path"

# Input folders
data_dir   = os.path.join(project_dir, "Data")
output_dir = os.path.join(project_dir, "Output")

# Pivot polygons
pivot_path = os.path.join(data_dir, "West_Minya_Pivots.geojson")

# WaPOR cubes
wapor_seasonal_cubes_path = os.path.join(output_dir, "Seasonal_cubes")
wapor_dekadal_cubes_path  = os.path.join(output_dir, "Dekadal_cubes")

# SAVI products
savi_pivots_csv_path          = os.path.join(output_dir, "SAVI_pivots_seasonal_0.3.csv")
savi_multi_seasonal_cube_path = os.path.join(output_dir, "SAVI_multi_seasonal_cube.nc")

# Final output folders
figures_dir = os.path.join(project_dir, "final_outputs", "figures")
tables_dir  = os.path.join(project_dir, "final_outputs", "tables")
rasters_dir = os.path.join(project_dir, "final_outputs", "rasters")

os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir,  exist_ok=True)
os.makedirs(rasters_dir, exist_ok=True)

# Build full paths to each seasonal .nc cube
aeti_nc = os.path.join(wapor_seasonal_cubes_path, "AETI_seasonal_total.nc")
npp_nc  = os.path.join(wapor_seasonal_cubes_path, "NPP_seasonal_total.nc")
pcp_nc  = os.path.join(wapor_seasonal_cubes_path, "PCP_seasonal_total.nc")

print("Project folders configured successfully.")

## 2 · Data Loading & Spatial Integrity Validation

WaPOR L3 seasonal cubes (AETI, NPP, PCP) and pivot vector layer are loaded. Three hard assertions are enforced before any analysis proceeds:

1. **Resolution bounds** — raster pixel size must be within the WaPOR L3 operational range (5–100m)
2. **Grid alignment** — AETI, NPP, and PCP must share the same pixel grid after PCP resampling from 5km to 20m (applied in Notebook 1)
3. **CRS match** — pivot polygons and rasters must share EPSG:32636 (UTM Zone 36N)

A **−20m negative buffer** is applied to all pivot polygons for zonal extraction only. At WaPOR's 20m resolution this removes one full pixel ring, eliminating mixed-pixel contamination where crop canopy and surrounding desert sand co-occur within a single pixel. Original unbuffered geometries are retained for all area and volume calculations.

In [ ]:
# --- 1. Load WaPOR Seasonal Cubes (AETI, NPP, PCP) ---
aeti_da = xr.open_dataset(aeti_nc, chunks='auto')['AETI']
npp_da = xr.open_dataset(npp_nc, chunks='auto')['NPP']
pcp_da = xr.open_dataset(pcp_nc, chunks='auto')['PCP']

# --- 2. Grid Geometry Verification ---
expected_res = abs(aeti_da.rio.resolution()[0])
res_check = all(abs(r) == expected_res for da in [aeti_da, npp_da, pcp_da] for r in da.rio.resolution())

# --- 3. Pivot Vector Infrastructure ---
savi_raw = pd.read_csv(savi_pivots_csv_path)
savi_raw['geometry'] = savi_raw['geometry'].apply(wkt_loads)
pivots_gdf = gpd.GeoDataFrame(savi_raw, geometry='geometry', crs='EPSG:32636')

if 'pivot_id' not in pivots_gdf.columns:
    pivots_gdf = pivots_gdf.reset_index().rename(columns={'index': 'pivot_id'})
pivots_gdf['pivot_id'] = pivots_gdf['pivot_id'].astype(int)

# Retain original unbuffered geometries for true area volume assessments
pivots_gdf['area_ha'] = pivots_gdf.geometry.area / 10000.0

# Create an inward 20-meter negative buffer layer
# to eliminate mixed border pixels containing background desert sand noise
pivots_buffered_gdf = pivots_gdf.copy()
pivots_buffered_gdf['geometry'] = pivots_buffered_gdf.geometry.buffer(-20.0)

# --- 4. Spatial Alignment Validation ---
crs_match = pivots_gdf.crs.to_epsg() == aeti_da.rio.crs.to_epsg()

print(f'Infrastructure Loaded: {len(pivots_gdf)} pivots.')
print(f'Resolution: {expected_res}m | Grid Alignment: {res_check} | CRS Match: {crs_match}')
print("Vector infrastructure ready. Negative spatial buffering applied successfully.")

# Ensure data integrity before processing
assert 10 <= expected_res <= 100, "Resolution outside WaPOR v3 operational range."
assert res_check, "Raster grid misalignment detected."
assert crs_match, "CRS mismatch between vector and raster layers."

In [ ]:
pivots_gdf.head()

## 3 · SAVI-Derived Seasonal Activity Matrix

The binary activity matrix encodes which pivots were actively cultivated in each of the four monitored winter seasons. It is derived from Sentinel-2 SAVI composites computed in Notebook 3 (median SAVI ≥ 0.3 threshold).

This matrix serves as the master activity gate throughout the analysis. AETI alone cannot distinguish crop transpiration from bare-soil evaporation in a desert environment — the SAVI-based classification provides the independent vegetation signal needed to isolate true agronomic water consumption.

> Huete, A.R. (1988). A soil-adjusted vegetation index (SAVI). *Remote Sensing of Environment*, 25(3), 295–309.

In [ ]:
# Map multi-seasonal cultivation status into a binary activity matrix
status_cols = ['S1_Status', 'S2_Status', 'S3_Status', 'S4_Status']
activity_matrix = pivots_gdf[['pivot_id'] + status_cols].copy()

for col in status_cols:
    # Standardize string entries to binary status (1: Active, 0: Inactive)
    activity_matrix[col] = (activity_matrix[col].str.strip().str.lower() == 'active').astype(int)

activity_matrix = activity_matrix.set_index('pivot_id')
activity_matrix.columns = SEASON_LABELS
pivots_gdf['n_active_seasons'] = pivots_gdf['pivot_id'].map(activity_matrix.sum(axis=1))

print(f"Activity matrix generated for {len(activity_matrix)} units across {N_SEASONS} seasons.")

In [ ]:
# Dynamic figsize scaling for cleaner dashboard visualization
dynamic_height = min(len(activity_matrix) * 0.15, 25)
fig = plt.figure(figsize=(8, dynamic_height))

binary_cmap = mcolors.ListedColormap(['#e74c3c', '#27ae60'])
ax = sns.heatmap(activity_matrix, cmap=binary_cmap, cbar=True, linewidths=0.1, linecolor='white', cbar_kws={'shrink': 0.2})

cbar = ax.collections[0].colorbar
cbar.set_ticks([0.25, 0.75])
cbar.set_ticklabels(['Inactive', 'Active'])

plt.title("Seasonal Activity Status (2021–2025)")
plt.xlabel('Season')
plt.ylabel('Pivot ID')

# Export the heatmap to the figures folder
save_path = os.path.join(figures_dir, "pivot_activity_matrix.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')

print(f"Heatmap exported to: {save_path}")
plt.show()

## 4 · Raster Integrity Check & Zonal Extraction Function

A plausibility check confirms that all three raster variables contain physically valid, non-constant values after WaPOR v3 scale factor application. Constant rasters indicate a failed or corrupted export.

The `get_zonal_stats` function uses `rasterstats` with explicit NoData handling (`-9999`). This integer NoData value is more stable than NaN across different GDAL versions and prevents silent mishandling of missing pixels in zonal computations.

**Note on PCP scale factor:** WaPOR L3 PCP is delivered in native mm/dekad without integer compression — no scale factor is required. This differs from AETI (×10) and NPP (×1000), which are stored as scaled integers.

In [ ]:
# --- Diagnostic Check with Corruption Detection ---
print('--- Raster Integrity Check ---')
for name, da in [('AETI', aeti_da), ('NPP', npp_da), ('PCP', pcp_da)]:
    val_min = da.min().compute().item()
    val_max = da.max().compute().item()
    # Corruption Check: Detect constant rasters or failed exports
    assert val_max > val_min, f"Potential corrupted raster in {name}: Constant value detected."
    print(f'{name:4} | Min: {val_min:8.2f} | Max: {val_max:8.2f} | Integrity: OK')

# --- Zonal Statistics with NoData Stability ---
def get_zonal_stats(raster_da, polygons_gdf, var_name):
    # rasterstats handles explicit integer nodata more consistently than NaN
    raster_values = np.nan_to_num(raster_da.values, nan=-9999)

    stats = rasterstats.zonal_stats(
        polygons_gdf,
        raster_values,
        affine=raster_da.rio.transform(),
        stats=['mean'],
        nodata=-9999
    )
    return pd.DataFrame(stats).rename(columns={'mean': var_name})

print('\nZonal function and diagnostic integrity check complete.')

## 5 · Seasonal Zonal Extraction & Activity Masking

WaPOR v3 L3 scale factors are applied per season to recover physical units from raw integer storage:

| Variable | Scale Factor | Physical Unit |
|---|---|---|
| AETI | × 10.0 | mm / season |
| NPP | × 1000.0 | gC/m² / season |
| PCP | × 1.0 | mm / season (native) |

**Critical implementation — inactive-pivot zeroing:** Immediately after the SAVI activity gate is applied, `aeti_mm` and `npp_gC_m2` are set to zero for all inactive pivot-seasons. This ensures that all downstream biophysical transformations and volumetric aggregations are correctly zero for inactive units — regardless of how subsequent groupby operations are structured. Without this step, bare-soil evaporation from fallow desert fields would inflate groundwater demand estimates.

Zonal extraction uses the **20m negatively buffered** polygons to eliminate mixed edge pixels. Area and volume calculations use the **original unbuffered** polygons.

In [ ]:
# WaPOR v3 Scale Factors (FAO Manual 3.0)
AETI_CORRECTION = 10.0  # raw (0.1) -> mm
NPP_CORRECTION = 1000.0 # raw (0.001) -> gC/m2
PCP_CORRECTION = 1.0    # WaPOR PCP delivered in native mm — no scale factor needed

# Agro-Biophysical Conversion Constants
CARBON_TO_DRY_MATTER = 2.22
M2_TO_HA_KG = 10.0

seasonal_results = []
for i, label in enumerate(SEASON_LABELS):
    # Extract and scale
    aeti_s = aeti_da.isel(season=i).compute() * AETI_CORRECTION
    npp_s  = npp_da.isel(season=i).compute() * NPP_CORRECTION
    pcp_s  = pcp_da.isel(season=i).compute() * PCP_CORRECTION

    # Execute zonal extraction
    aeti_vals = get_zonal_stats(aeti_s, pivots_buffered_gdf, 'aeti_mm')
    npp_vals  = get_zonal_stats(npp_s,  pivots_buffered_gdf, 'npp_gC_m2')
    pcp_vals  = get_zonal_stats(pcp_s,  pivots_buffered_gdf, 'pcp_mm')

    df = pivots_gdf[['pivot_id', 'area_ha']].copy()
    df['season'] = label
    df['aeti_mm'] = aeti_vals['aeti_mm']
    df['npp_gC_m2'] = npp_vals['npp_gC_m2']
    df['pcp_mm'] = pcp_vals['pcp_mm']

    # Apply activity mask early to prevent volumetric inflation
    df['is_active'] = df['pivot_id'].map(activity_matrix[label])

    # CRITICAL FIX: If not active, zero out consumption to prevent demand inflation
    mask = df['is_active'] == 0
    df.loc[mask, 'aeti_mm'] = 0.0
    df.loc[mask, 'npp_gC_m2'] = 0.0

    # BIOPHYSICAL TRANSFORMATIONS
    df['biomass_dry_kg_ha'] = df['npp_gC_m2'] * M2_TO_HA_KG * CARBON_TO_DRY_MATTER
    df['water_consumed_m3_ha'] = df['aeti_mm'] * 10.0

    seasonal_results.append(df)

stats_df = pd.concat(seasonal_results, ignore_index=True)
print(f"Parsed stats for {len(stats_df)} pivot-seasons with inactive-row zeroing applied.")

## 6 · Biomass Water Productivity (WP$_b$)

**Gross Biomass Water Productivity** is the primary performance metric of this analysis, computed in volumetric units (kg/m³) consistent with the FAO WaPOR benchmarking framework:

$$WP_b \ (kg/m^3) = \frac{\text{Total Dry Matter Biomass} \ (kg/ha)}{\text{Actual Water Consumed} \ (m^3/ha)}$$

**Carbon-to-Dry-Matter conversion:**
Plant dry matter contains approximately 45% carbon by mass. WaPOR NPP is reported in gC/m² — conversion to dry matter biomass (kg/ha) applies:

$$\text{Biomass} \ (kg/ha) = NPP \ (gC/m^2) \times \frac{1}{0.45} \times 10 = NPP \times 2.22 \times 10$$

The ×10 factor converts g/m² to kg/ha (10,000 m²/ha ÷ 1,000 g/kg).

**Benchmark:** FAO WaPOR arid-zone irrigated agriculture reference range = **0.8–1.8 kg/m³**

> FAO (2023). *WaPOR Database Methodology: Version 3.* Food and Agriculture Organization, Rome.
> IHE Delft (2020). *Water Productivity and Water Accounting using WaPOR.*

In [ ]:
# --- Standard Volumetric Crop Water Productivity (CWP) Calculation ---
# Formula: Total Gross Dry Biomass (kg/ha) / Volumetric Net Water Consumption (m3/ha)
# Note: We use 'water_consumed_m3_ha' as defined in the seasonal extraction cell.
stats_df['cwp'] = stats_df['biomass_dry_kg_ha'] / stats_df['water_consumed_m3_ha'].replace(0, np.nan)

print("Volumetric CWP (kg/m³) calculated successfully using net water consumption.")

### 6.1 · Seasonal WP$_b$ Distribution — Benchmark Comparison

Boxplots show the distribution of WP$_b$ across active pivot-seasons per year. Individual pivot observations are overlaid as strip points. Two FAO WaPOR reference lines mark the arid-zone benchmark range (0.8–1.8 kg/m³).

**Interpretation note:**

**Crop context:** Field knowledge indicates sugar beet (*Beta vulgaris*) as
the dominant crop in this cluster, followed by wheat, clover, and other
winter crops. The brackish irrigation water (TDS 2,176–2,912 mg/L, Khalil
et al. 2024) is consistent with sugar beet dominance, as it is one of the
most salt-tolerant field crops. WaPOR WPb measures total dry matter biomass
— including storage roots, leaves, and stems — which is substantially higher
for sugar beet than for cereals, providing a scientifically coherent
explanation for the above-benchmark project mean WPb of 2.054 kg/m³..

In [ ]:
plt.figure(figsize=(12, 7))

# Filter for active pivots with valid CWP data
active_cwp_df = stats_df[(stats_df['is_active'] == 1) & (stats_df['cwp'].notna())].copy()

# Main boxplot
sns.boxplot(data=active_cwp_df, x='season', y='cwp', palette='YlGnBu', hue='season', legend=False, showfliers=False)

# Add individual pivot points for granularity
sns.stripplot(data=active_cwp_df, x='season', y='cwp', color='black', alpha=0.25, size=2, jitter=True)

# Add Benchmark Lines (Converted to kg/m3 equivalents for biomass)
plt.axhline(0.8, color='orange', linestyle='--', alpha=0.6, label='Lower Benchmark (0.8 kg/m³)')
plt.axhline(1.8, color='green', linestyle='--', alpha=0.6, label='Upper Benchmark (1.8 kg/m³)')
plt.axhline(active_cwp_df['cwp'].mean(), color='red', linestyle='-', linewidth=1, label=f'Project Mean ({active_cwp_df["cwp"].mean():.2f})')

plt.title('Seasonal Biomass Water Productivity (WPb) Benchmarking\n[Filtered for Active Center Pivots]', pad=20)
plt.ylabel('Water Productivity (kg/m³)')
plt.xlabel('Agricultural Season')
plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
plt.grid(axis='y', linestyle=':', alpha=0.5)

# Save as a distinct deliverable asset
cwp_active_plot_path = os.path.join(figures_dir, "seasonal_wpb_active_only_boxplot.png")
plt.savefig(cwp_active_plot_path, dpi=300, bbox_inches='tight')

plt.show()
print(f"Active WPb boxplot saved to: {cwp_active_plot_path}")

## 7 · Groundwater Demand Estimation

In hyper-arid environments such as West Minya (annual rainfall < 5mm), irrigation demand is driven almost entirely by the AETI flux. The estimation chain follows three steps:

#### Step 1 — Effective Rainfall
Not all precipitation reaches the root zone. Surface runoff and deep percolation reduce availability. A fixed 80% coefficient is applied, consistent with FAO CROPWAT methodology for small rainfall events in arid zones:

$$P_{eff} = P \times 0.80$$

#### Step 2 — Net Irrigation Water Requirement
$$NIWR \ (mm) = AETI - P_{eff} \quad (\text{clipped at } 0)$$

#### Step 3 — Gross Groundwater Abstraction
Vertical depth is converted to volumetric extraction, accounting for center-pivot system losses (spray evaporation, wind drift, distribution non-uniformity). Modern center-pivots operate at 80–90% application efficiency; η = 0.85 is used as a conservative benchmark:

$$\text{Abstraction} \ (m^3) = \frac{(NIWR / 1000) \times (A_{ha} \times 10{,}000)}{\eta_i = 0.85}$$

**AETI vs Abstraction:** AETI is the water consumed by the crop at the field boundary. Abstraction is the total volume extracted from the aquifer — always larger due to system losses.

> Allen et al. (1998). *FAO Irrigation and Drainage Paper 56.*

In [ ]:
# ---Volumetric Calculation ---
# NIWR (mm) = AETI - (PCP * 0.8).
# Note: aeti_mm is already 0 for inactive pivots from the extraction step.
stats_df['niwr_mm'] = (stats_df['aeti_mm'] - (stats_df['pcp_mm'] * EFF_RAINFALL_COEFF)).clip(lower=0)

# If aeti was 0 (inactive), niwr should strictly be 0 (no irrigation occurred)
stats_df.loc[stats_df['is_active'] == 0, 'niwr_mm'] = 0.0

# Demand Volume (m3)
stats_df['demand_vol_m3'] = (stats_df['niwr_mm'] / 1000.0) * (stats_df['area_ha'] * 10000.0)
stats_df['abstraction_vol_m3'] = stats_df['demand_vol_m3'] / APPLICATION_EFFICIENCY

# Summarize
seasonal_vol = (
    stats_df.groupby('season')['abstraction_vol_m3']
    .sum()
    .reset_index()
)
seasonal_vol['vol_mcm'] = seasonal_vol['abstraction_vol_m3'] / 1e6

print('\n=== Seasonal Irrigation Water Demand (MCM) ===')
print(seasonal_vol[['season', 'vol_mcm']].to_string(index=False))

### 7.1 · Seasonal Groundwater Demand (MCM)

Total estimated groundwater abstraction per season in Million Cubic Meters (MCM), aggregated from active pivots only. The monotonic increase from 13.5 MCM (2021/22) to 28.3 MCM (2024/25) tracks closely with the documented area expansion.

**Calendar Translation for Irrigation Season (Dekads 8–17):**
*   **Start:** Dekad 8 = **late January**.
*   **End:** Dekad 17 = **mid-to-late April**.
*   **Duration:** The active window remains stable at 10 dekads (100 days).

In [ ]:
plt.figure(figsize=(10, 6))

# Create the bar plot for seasonal volumes
ax = sns.barplot(data=seasonal_vol, x='season', y='vol_mcm', palette='Blues_d', hue='season', legend=False)

# Annotate bars with the actual MCM value for reporting clarity
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points',
                fontsize=11, fontweight='bold')

plt.title('Total Estimated Seasonal Irrigation Demand (2021–2025)')
plt.ylabel('Volume (Million Cubic Meters - MCM)')
plt.xlabel('Winter Season')
plt.ylim(0, seasonal_vol['vol_mcm'].max() * 1.2) # Add headroom for labels
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Save as a project deliverable
demand_plot_path = os.path.join(figures_dir, "seasonal_irrigation_demand_mcm.png")
plt.savefig(demand_plot_path, dpi=300, bbox_inches='tight')

plt.show()
print(f"Seasonal demand chart saved to: {demand_plot_path}")

### 7.2 · Mean Seasonal Water Consumption per Active Pivot (AETI)

Mean AETI per active pivot with ±1 standard deviation error bars. The lower mean in 2022/23 (362 mm) reflects the new expansion cohort entering the dataset in their first operational season — not a drought signal. The narrowing standard deviation from 2022/23 onward suggests the expanding cluster is becoming more homogeneous in its water use patterns, consistent with standardised pivot management across the operation.

**Note on 2021/22 comparability:** The active irrigation window in 2021/22 was 7 dekads (70 days) versus 10 dekads (100 days) in subsequent seasons. The 452 mm seasonal total reflects ~30% less of the growing season and is not directly comparable to later years.

In [ ]:
plt.figure(figsize=(10, 6))

# Filter for active pivots only
active_stats = stats_df[stats_df['is_active'] == 1]

# Create bar plot with standard deviation error bars
sns.barplot(
    data=active_stats,
    x='season',
    y='aeti_mm',
    palette='YlGnBu_d',
    errorbar='sd',
    capsize=.1,
    hue='season',
    legend=False
)

plt.title('Mean Seasonal Water Consumption (AETI) per Active Pivot')
plt.ylabel('AETI (mm / season)')
plt.xlabel('Winter Season')
plt.grid(axis='y', linestyle=':', alpha=0.5)

# Save as a distinct deliverable
aeti_pivot_plot_path = os.path.join(figures_dir, "seasonal_mean_aeti_active_pivots.png")
plt.savefig(aeti_pivot_plot_path, dpi=300, bbox_inches='tight')

plt.show()
print(f"Seasonal AETI bar chart saved to: {aeti_pivot_plot_path}")

## 8 · Water Productivity Trend Analysis — Mann-Kendall

**Sen's Slope** (median rate of change in WP$_b$ per season) is computed per pivot using the Mann-Kendall non-parametric test. Mann-Kendall is appropriate here because it requires no assumption of normality — an assumption that cannot be validated with $n = 4$ observations.

**Significance threshold:** $p < 0.1$ (applied and disclosed). With only 4 seasonal observations, statistical power is inherently low. No pivot is expected to reach $p < 0.05$. Sen's Slope is therefore used as a **directional performance indicator** rather than a confirmed trend.

**Scope:** Trend analysis is applied to all pivots with ≥ 3 active seasons. The slope distribution chart is restricted to operationally active classes (Expansion, Stable Active, Intermittent/Rotation), excluding Abandonment and Permanent Fallow pivots whose trend signals are not agronomically interpretable.

> Mann, H.B. (1945). Nonparametric tests against trend. *Econometrica*, 13(3), 245–259.
> Sen, P.K. (1968). Estimates of the regression coefficient based on Kendall's tau. *JASA*, 63(324), 1379–1389.

In [ ]:
def calculate_cwp_trend(group):
    values = group['cwp'].dropna().values
    if len(values) < 3:
        return pd.Series({'trend': 'Insufficient Data', 'p_value': np.nan, 'slope': np.nan})
    try:
        res = mk.original_test(values)
        final_trend = res.trend if res.p < 0.1 else 'no trend'
        return pd.Series({'trend': final_trend, 'p_value': res.p, 'slope': res.slope})
    except Exception:
        return pd.Series({'trend': 'Error', 'p_value': np.nan, 'slope': np.nan})

trend_results = stats_df.groupby('pivot_id').apply(calculate_cwp_trend, include_groups=False).reset_index()

if 'trend' in pivots_gdf.columns:
    pivots_gdf = pivots_gdf.drop(columns=['trend', 'p_value', 'slope'])

pivots_gdf = pivots_gdf.merge(trend_results, on='pivot_id', how='left')
print('Trend analysis successfully merged into pivots_gdf.')

### 8.1 · Operational Classification & Multi-Season Average WP$_b$

Each pivot is assigned an operational class based on its 4-season activity pattern. Average WP$_b$ is computed from **active seasons only** — inactive seasons are excluded to prevent bare-soil evaporation from contaminating the efficiency score.

The top 10 pivots by average WP$_b$ are displayed for benchmarking reference.

In [ ]:
# 1. Ensure CWP is calculated in the main dataframe
# Formula: Total Gross Dry Biomass (kg/ha) / Volumetric Net Water Consumption (m3/ha)
stats_df['cwp'] = stats_df['biomass_dry_kg_ha'] / stats_df['water_consumed_m3_ha'].replace(0, np.nan)

# 2. Define operational classification logic
# Classification rules:
# Abandonment = active in S1 but inactive in S4, regardless of S2/S3
# A pivot active in 3 of 4 seasons but idle in S4 is still flagged as Abandonment.
# This reflects project-end status, not overall utilization rate.
def classify_operation(row):
    pid = row['pivot_id']
    s12_inactive = (activity_matrix.loc[pid, SEASON_LABELS[0:2]].sum() == 0)
    s34_active   = (activity_matrix.loc[pid, SEASON_LABELS[2:4]].sum() >= 1)
    is_s1_active = activity_matrix.loc[pid, SEASON_LABELS[0]] == 1
    is_s4_active = activity_matrix.loc[pid, SEASON_LABELS[3]] == 1

    if s12_inactive and s34_active: return 'Expansion (New)'
    if is_s1_active and not is_s4_active: return 'Abandonment'
    if row['n_active_seasons'] >= 3: return 'Stable Active'
    if row['n_active_seasons'] == 0: return 'Permanent Fallow'
    return 'Intermittent / Rotation'

pivots_gdf['operation_class'] = pivots_gdf.apply(classify_operation, axis=1)

# 3. Calculate average WPb (cwp) for each pivot across only ACTIVE seasons
pivot_avg_wpb = (stats_df[stats_df['is_active'] == 1]
                 .groupby('pivot_id')['cwp']
                 .mean()
                 .reset_index())
pivot_avg_wpb.columns = ['pivot_id', 'avg_wpb_kg_m3']

# 4. Merge into the main GeoDataFrame
if 'avg_wpb_kg_m3' in pivots_gdf.columns:
    pivots_gdf = pivots_gdf.drop(columns=['avg_wpb_kg_m3'])

pivots_gdf = pivots_gdf.merge(pivot_avg_wpb, on='pivot_id', how='left')

# 5. Display the top 10 most efficient pivots on average
display(pivots_gdf[['pivot_id', 'avg_wpb_kg_m3', 'operation_class']].sort_values('avg_wpb_kg_m3', ascending=False).head(10))

### 8.2 · Distribution of Average WP$_b$ Across All Pivots

The histogram reveals a **bimodal structure** rather than a normal distribution — two distinct pivot populations are present:
- **Lower cluster (1.4–1.9 kg/m³):** Expansion and Intermittent pivots in early operational stages
- **Upper cluster (2.4–2.9 kg/m³):** Stable Active pivots with multiple productive seasons

The project mean (1.98 kg/m³) falls in the gap between both populations and does not accurately characterise either group. The median (1.84 kg/m³) is a more representative central tendency for the lower cluster. Both statistics should be reported together.

In [ ]:
plt.figure(figsize=(10, 6))

# Create distribution plot for the average WPb column
sns.histplot(pivots_gdf['avg_wpb_kg_m3'], kde=True, color='forestgreen', bins=20)

# Add project mean line
plt.axvline(pivots_gdf['avg_wpb_kg_m3'].mean(), color='red', linestyle='--', label=f"Mean: {pivots_gdf['avg_wpb_kg_m3'].mean():.2f} kg/m³")

plt.title('Distribution of Average Multi-Seasonal Water Productivity (WPb)', fontsize=14, fontweight='bold')
plt.xlabel('Average WPb (kg/m³)', fontsize=12)
plt.ylabel('Number of Pivots', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Save the distribution figure
avg_wpb_dist_path = os.path.join(figures_dir, 'average_wpb_distribution.png')
plt.savefig(avg_wpb_dist_path, dpi=300)

plt.show()
print(f'Average WPb distribution plot saved to: {avg_wpb_dist_path}')

### 8.3 · Sen's Slope Distribution — Operationally Active Pivots

Distribution of the Sen's Slope for Expansion, Stable Active, and Intermittent pivots only ($n = 40$). Positive values indicate improving WP$_b$ over the monitoring period; negative values indicate declining efficiency. The mean slope and the proportion of pivots on each side of zero are the primary diagnostic outputs.

In [ ]:
plt.figure(figsize=(10, 6))
# Final consolidated trend analysis for active pivots
active_mask = pivots_gdf['operation_class'].isin(['Expansion (New)', 'Stable Active', 'Intermittent / Rotation'])
active_slopes = pivots_gdf[active_mask & pivots_gdf['slope'].notna()]

if not active_slopes.empty:
    sns.histplot(active_slopes['slope'], kde=True, color='darkcyan', bins=15)
    plt.axvline(0, color='red', linestyle='--', alpha=0.7, label='No Change')
    plt.axvline(active_slopes['slope'].mean(), color='navy', linestyle='-', label=f'Mean Slope: {active_slopes["slope"].mean():.3f}')
    plt.title('Distribution of Efficiency Slopes (Active Units)', fontsize=14, fontweight='bold')
    plt.xlabel('kg/m³ change per season')
    plt.ylabel('Pivot Count')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.savefig(os.path.join(figures_dir, 'cwp_trend_slopes_active.png'), dpi=300, bbox_inches='tight')
    plt.show()

## 9 · Spatial Distribution of Operational Trajectories

The map below assigns each of the 166 pivots a colour corresponding to its operational class, positioned at its true geographic location. Spatial clustering of classes reveals structural patterns in how development is progressing across the cluster — where expansion fronts are active, where the operational core is concentrated, and where instability is concentrated.

---

### Operational Classification — Definitions

All classes reflect observed activity within the **2021–2025 monitoring window only**, based on a median SAVI threshold of 0.3. No pivot history prior to Season 1 (2021/22) is described.

| Class | Rule | Interpretation |
|---|---|---|
| **Stable Active** | Active in ≥ 3 of 4 seasons | Consistently productive core |
| **Expansion (New)** | Inactive S1+S2; active in S3 or S4 | New or recommissioned units |
| **Abandonment** | Active S1; inactive S4 | Transition to non-use (end-state rule) |
| **Intermittent / Rotation** | 1–2 active seasons, no clear direction | Irregular or rotating cultivation |
| **Permanent Fallow** | Zero active seasons | Idle infrastructure across full period |

> **Edge case:** A pivot active in S1, S2, S3 but idle in S4 is classified as Abandonment. This rule prioritises project-end status over overall utilisation rate. The classification is applied consistently and disclosed here.

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Define Qualitative Palette
class_colors = {
    'Expansion (New)':     '#2ecc71',
    'Stable Active':       '#3498db',
    'Abandonment':         '#e74c3c',
    'Intermittent / Rotation': '#f39c12',
    'Permanent Fallow':    '#95a5a6'
}

pivots_gdf['color'] = pivots_gdf['operation_class'].map(class_colors)

# Plot using manual colors
pivots_gdf.plot(color=pivots_gdf['color'], ax=ax, edgecolor='black', linewidth=0.3)

# Build manual legend
legend_patches = [mpatches.Patch(color=color, label=label) for label, color in class_colors.items()]
ax.legend(handles=legend_patches, title="Operational Class", loc='upper left', bbox_to_anchor=(1, 1))

plt.title("Spatial Distribution of Pivot Operational Trajectories (2021-2025)")
ax.set_axis_off()

spatial_map_path = os.path.join(figures_dir, "spatial_operational_classification.png")
plt.savefig(spatial_map_path, dpi=300, bbox_inches='tight')
plt.show()

## 10 · Seasonal Performance Summary

Project-wide aggregates computed from **active pivot-seasons only**. The table presents the four core monitoring metrics — cultivated area, mean and median water productivity, and total groundwater abstraction — across the four monitored winter seasons.

In [ ]:
# --- Seasonal Performance Summary Table (Water Productivity) ---
seasonal_summary = stats_df[stats_df['is_active'] == 1].groupby('season').agg({
    'pivot_id': 'count',
    'area_ha': 'sum',
    'cwp': ['mean', 'median'],
    'abstraction_vol_m3': 'sum'
}).reset_index()

# Flatten multi-index columns and assign professional names
seasonal_summary.columns = ['Season', 'Active_Pivots', 'Total_Area_Ha', 'Mean_WP', 'Median_WP', 'Demand_Volume_m3']
seasonal_summary['Total_Volume_MCM'] = seasonal_summary['Demand_Volume_m3'] / 1e6

display(seasonal_summary.round(2))

## 11 · Quantitative Synthesis of Findings

The following block computes and reports all key performance indicators derived from the analysis. Values are calculated dynamically from the analytical outputs — not hardcoded — ensuring consistency with the full pipeline.

In [ ]:
# --- Expanded Quantitative Synthesis ---

# 1. Expansion Details
expansion_count = (pivots_gdf['operation_class'] == 'Expansion (New)').sum()
start_area = seasonal_summary.iloc[0]['Total_Area_Ha']
end_area = seasonal_summary.iloc[-1]['Total_Area_Ha']
area_growth_pct = ((end_area - start_area) / start_area) * 100

# 2. Water & Productivity Metrics
peak_mcm = seasonal_summary['Total_Volume_MCM'].max()
active_stats = stats_df[stats_df['is_active'] == 1]
mean_wpb = active_stats['cwp'].mean()
median_wpb = active_stats['cwp'].median()

# 3. Operational Stability
stable_pivots = (pivots_gdf['operation_class'] == 'Stable Active').sum()
abandoned_pivots = (pivots_gdf['operation_class'] == 'Abandonment').sum()

print('\n' + '='*75)
print('DETAILED PROJECT FINDINGS: WEST MINYA RECLAMATION (2021-2025)')
print('='*75)
print(f'\u2022 LAND DYNAMICS:')
print(f'  - Total Area Expansion: {start_area:,.0f} ha (2021) \u27a1 {end_area:,.0f} ha (2025)')
print(f'  - Growth Rate: +{area_growth_pct:.1f}% increase in cultivated footprint.')
print(f'  - Pivot Status: {expansion_count} expansions, {stable_pivots} stable, {abandoned_pivots} abandoned.')
print(f'\n\u2022 HYDROLOGICAL IMPACT:')
print(f'  - Peak Groundwater Demand: {peak_mcm:.1f} MCM (reached in Season 2024/25).')
print(f'  - Total Units Monitored: {len(pivots_gdf)} center-pivots.')
print(f'\n\u2022 PRODUCTIVITY BENCHMARKS:')
print(f'  - Project Mean WPb: {mean_wpb:.2f} kg/m³ (Median: {median_wpb:.2f} kg/m³).')
print(f'  - FAO Benchmark Status: Performance is ~14% above upper standard (1.8 kg/m³).')
print('='*75)

## 12 · Operational Synthesis — Multi-Panel Summary Figure

A four-panel figure integrating the primary spatial and quantitative findings: geographic operational trajectories, cultivated area growth, seasonal groundwater demand, and water productivity benchmarking. Designed as a standalone reporting figure suitable for technical presentations and project documentation.

In [ ]:
# Set professional context
plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(24, 18))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

class_colors = {
    'Expansion (New)':     '#2ecc71',
    'Stable Active':       '#3498db',
    'Abandonment':         '#e74c3c',
    'Intermittent / Rotation': '#f39c12',
    'Permanent Fallow':    '#95a5a6'
}

# --- Panel 1: Spatial Map ---
ax1 = fig.add_subplot(gs[0:2, 0])
pivots_gdf['color'] = pivots_gdf['operation_class'].map(class_colors)
pivots_gdf.plot(color=pivots_gdf['color'], ax=ax1, edgecolor='black', linewidth=0.2)

# Build manual legend
legend_patches = [mpatches.Patch(color=color, label=label) for label, color in class_colors.items()]
ax1.legend(handles=legend_patches, title='Operational Class', loc='lower left', fontsize=10)

ax1.set_title("I. GEOSPATIAL OPERATIONAL TRAJECTORY", fontsize=16, fontweight='bold')
ax1.set_axis_off()

# --- Panel 2: Area Expansion ---
ax2 = fig.add_subplot(gs[0, 1])
sns.barplot(data=seasonal_summary, x='Season', y='Total_Area_Ha', palette='YlGn_r', ax=ax2, hue='Season', legend=False)
ax2.set_title("II. CULTIVATED AREA GROWTH (HA)", fontsize=14, fontweight='bold')

# --- Panel 3: Volumetric Demand ---
ax3 = fig.add_subplot(gs[1, 1])
sns.barplot(data=seasonal_summary, x='Season', y='Total_Volume_MCM', palette='Blues_r', ax=ax3, hue='Season', legend=False)
ax3.set_title("III. SEASONAL WATER DEMAND (MCM)", fontsize=14, fontweight='bold')

# --- Panel 4: WPb Benchmarking ---
ax4 = fig.add_subplot(gs[0:2, 2])
active_df = stats_df[stats_df['is_active']==1]
sns.boxplot(data=active_df, x='season', y='cwp', palette='RdYlGn', ax=ax4, hue='season', legend=False, showfliers=False)
ax4.axhline(0.8, color='orange', linestyle='--', alpha=0.6, label='Benchmark Min (0.8)')
ax4.axhline(1.8, color='green', linestyle='--', alpha=0.6, label='Benchmark Max (1.8)')
ax4.set_title("IV. BIOMASS WATER PRODUCTIVITY (WPb)", fontsize=16, fontweight='bold')
ax4.legend(loc='upper right')

# --- Panel 5: Key Performance Indicators ---
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')
summary_text = (f"PROJECT SUMMARY: WEST MINYA (2021-2025)\n"
                f"• Expansion: {expansion_count} New Pivots | {area_growth_pct:.1f}% Growth\n"
                f"• Resource Load: {peak_mcm:.1f} MCM Peak Demand\n"
                f"• Efficiency: {active_df['cwp'].mean():.2f} Mean WPb (Target: 0.8-1.8 kg/m³ per FAO WaPOR)")
ax5.text(0.5, 0.5, summary_text, ha='center', va='center', fontsize=22, fontweight='bold', bbox=dict(facecolor='white', alpha=0.8, edgecolor='navy', boxstyle='round,pad=1'))

plt.suptitle("AGRICULTURAL MONITORING & BIOMASS PRODUCTIVITY DASHBOARD", fontsize=28, fontweight='black', y=0.98)
plt.show()

## 13 · Spatial Evolution of Water Productivity — Four-Season Grid

A 2×2 panel grid showing the spatial distribution of WP$_b$ (kg/m³) for each monitored season. Inactive pivots are shown in grey as spatial context. The consistent colorbar (5th–95th percentile range) enables direct cross-season comparison of productivity levels and their geographic distribution across the cluster.

**Key observation:** High-WP$_b$ performance concentrates in specific geographic sub-zones consistently across seasons, suggesting spatially structured variation in soil quality, water access, or management experience within the cluster.

In [ ]:
# Prepare data for spatial plotting
# We need to merge the long-form stats_df back to the geometry for each season
seasonal_geodata = []
for season in SEASON_LABELS:
    season_df = stats_df[(stats_df['season'] == season) & (stats_df['is_active'] == 1)]
    gdf_merged = pivots_gdf[['pivot_id', 'geometry']].merge(season_df, on='pivot_id')
    seasonal_geodata.append(gdf_merged)

# Configure 2x2 Plot
fig, axes = plt.subplots(2, 2, figsize=(20, 18))
axes = axes.flatten()

# Define consistent color scale based on the 5th and 95th percentiles of active CWP
vmin = stats_df[stats_df['is_active'] == 1]['cwp'].quantile(0.05)
vmax = stats_df[stats_df['is_active'] == 1]['cwp'].quantile(0.95)

for i, season in enumerate(SEASON_LABELS):
    ax = axes[i]
    # Plot all pivots in light grey as background
    pivots_gdf.plot(ax=ax, color='#eeeeee', edgecolor='#cccccc', linewidth=0.5)

    # Plot active pivots with WPb color mapping
    seasonal_geodata[i].plot(
        column='cwp',
        ax=ax,
        cmap='RdYlGn',
        vmin=vmin,
        vmax=vmax,
        edgecolor='black',
        linewidth=0.4
    )

    ax.set_title(f"Season: {season}", fontsize=16, fontweight='bold')
    ax.set_axis_off()

# Add a unified colorbar
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(vmin=vmin, vmax=vmax))
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Water Productivity (kg/m³)', fontsize=14, fontweight='bold')

plt.suptitle("Spatial Evolution of Volumetric Water Productivity (WPb)\nWest Minya Center Pivots (2021-2025)", fontsize=22, fontweight='black', y=0.98)

# Save the high-resolution deliverable
spatial_wpb_grid_path = os.path.join(figures_dir, 'spatial_wpb_4_season_grid.png')
plt.savefig(spatial_wpb_grid_path, dpi=300, bbox_inches='tight')

plt.show()
print(f"Spatial 4-season grid saved to: {spatial_wpb_grid_path}")

## 14 · Intra-Seasonal Phenological Analysis — Dekadal Resolution

Seasonal totals aggregate all crop development stages into a single value. Dekadal (10-day) resolution resolves the within-season dynamics of water consumption and biomass accumulation, enabling three complementary analyses:

1. **Phenological profiles** — timing of AETI and NPP peaks relative to the November 1 season start, with inter-pivot variability (±1 SD). Based on the curves, NPP peaks at dekads 13–16 (March–mid April), consistent with the storage root bulking and canopy maturation of sugar beet — the dominant crop in this cluster — followed by wheat, clover, and other winter cultivars. The brackish irrigation water (TDS 2,176–2,912 mg/L) constrains crop selection toward salt-tolerant species such as sugar beet (Khalil et al., 2024; Steduto et al., 2012).

2. **Season length** — first and last dekad where mean AETI ≥ 20 mm/10-days, above the bare-soil evaporation baseline.
3. **Cross-season consistency** — comparison of growth curves across four years to detect shifts in sowing dates or crop development patterns.

**Why WPb is not computed at dekadal resolution:** The biomass/water ratio is physically unstable within a single crop cycle. During germination, NPP approaches zero while AETI remains moderate — the ratio produces meaningless values. AETI and NPP are presented separately at this temporal scale to preserve physical interpretability.

**2021/22 comparability note:** The active irrigation window in 2021/22 is 7 dekads (starting dekad 11) versus 10 dekads (starting dekad 8) in all subsequent seasons — approximately 30% shorter.

In [ ]:
# Path to Multi-Seasonal Dekadal Cubes
dekadal_folder = os.path.join(output_dir, "Dekadal_Cubes")
aeti_dek_nc = os.path.join(dekadal_folder, "AETI_dekadal_cube.nc")
npp_dek_nc = os.path.join(dekadal_folder, "NPP_dekadal_cube.nc")

In [ ]:
# Load dekadal data
ds_aeti_dek = xr.open_dataset(aeti_dek_nc, chunks='auto')
ds_npp_dek  = xr.open_dataset(npp_dek_nc,  chunks='auto')

records = []
affine = ds_aeti_dek.rio.transform()

for s_idx, label in enumerate(SEASON_LABELS):
    for d_idx in range(len(ds_aeti_dek.dekad)):

        aeti_slice = ds_aeti_dek['AETI'].isel(season=s_idx, dekad=d_idx).values * AETI_CORRECTION
        npp_slice  = ds_npp_dek['NPP'].isel(season=s_idx, dekad=d_idx).values * NPP_CORRECTION

        a_stats = rasterstats.zonal_stats(pivots_buffered_gdf, aeti_slice,
                                          affine=affine, stats=['mean'], nodata=-9999)
        n_stats = rasterstats.zonal_stats(pivots_buffered_gdf, npp_slice,
                                          affine=affine, stats=['mean'], nodata=-9999)

        for i, (ar, nr) in enumerate(zip(a_stats, n_stats)):
            pid = pivots_gdf.iloc[i]['pivot_id']
            # Use the SAVI activity matrix — same gate as seasonal analysis
            is_active = activity_matrix.loc[pid, label] == 1
            if is_active and ar['mean'] is not None:
                records.append({
                    'season':    label,
                    'dekad_idx': d_idx,
                    'pivot_id':  pid,
                    'aeti_mm':   ar['mean'],       # mm/dekad
                    'npp_gC_m2': nr['mean'],       # gC/m²/dekad
                })

df_dek = pd.DataFrame(records)
print(f"Dekadal records: {len(df_dek)} | Seasons: {df_dek['season'].nunique()}")

In [ ]:
# AETI phenology per season
# The core irrigation scheduling diagnostic

summary_dek = df_dek.groupby(['season', 'dekad_idx']).agg(
    aeti_mean=('aeti_mm', 'mean'),
    aeti_std=('aeti_mm', 'std'),
    npp_mean=('npp_gC_m2', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette = sns.color_palette("Set1", n_colors=4)

# Left: AETI with uncertainty band
ax = axes[0]
for i, (season, grp) in enumerate(summary_dek.groupby('season')):
    ax.plot(grp['dekad_idx'], grp['aeti_mean'], label=season,
            color=palette[i], linewidth=2.5)
    ax.fill_between(grp['dekad_idx'],
                    grp['aeti_mean'] - grp['aeti_std'],
                    grp['aeti_mean'] + grp['aeti_std'],
                    color=palette[i], alpha=0.12)

ax.set_title('Seasonal Water Consumption Pattern (AETI)', fontweight='bold')
ax.set_xlabel('Dekad Index (Nov 1 = 0)')
ax.set_ylabel('Mean AETI (mm / 10 days)')
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.5)

# Right: NPP development curve
ax = axes[1]
for i, (season, grp) in enumerate(summary_dek.groupby('season')):
    ax.plot(grp['dekad_idx'], grp['npp_mean'], label=season,
            color=palette[i], linewidth=2.5)

ax.set_title('Seasonal Biomass Production Pattern (NPP)', fontweight='bold')
ax.set_xlabel('Dekad Index (Nov 1 = 0)')
ax.set_ylabel('Mean NPP (gC / m² / 10 days)')
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.5)

plt.suptitle('Multi-Seasonal Phenological Profiles — Active Pivots Only',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'phenology_aeti_npp_profiles.png'),
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Growing season length per season
# First and last dekad where mean AETI > 20mm (meaningful irrigation activity)

IRRIGATION_THRESHOLD_MM = 20.0  # mm/dekad — above bare-soil baseline

season_windows = []
for season, grp in summary_dek.groupby('season'):
    active_dekads = grp[grp['aeti_mean'] >= IRRIGATION_THRESHOLD_MM]['dekad_idx']
    if len(active_dekads) > 0:
        season_windows.append({
            'season':    season,
            'start_dek': active_dekads.min(),
            'end_dek':   active_dekads.max(),
            'length':    active_dekads.max() - active_dekads.min() + 1
        })

# Create DataFrame and add calculated columns
windows_df = pd.DataFrame(season_windows)
windows_df['length_days'] = windows_df['length'] * 10

# Identify the dekad index of peak AETI for each season
peak_dekads = summary_dek.loc[summary_dek.groupby('season')['aeti_mean'].idxmax()][['season','dekad_idx']].rename(columns={'dekad_idx':'peak_dekad'})
windows_df = windows_df.merge(peak_dekads, on='season')

print(windows_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
for _, row in windows_df.iterrows():
    ax.barh(row['season'],
            row['length'],
            left=row['start_dek'],
            height=0.5,
            color='steelblue', alpha=0.75, edgecolor='navy')
    ax.text(row['start_dek'] + row['length'] / 2,
            row['season'],
            f"{int(row['length'])} dekads",
            ha='center', va='center', color='white', fontweight='bold', fontsize=11)

ax.set_xlabel('Dekad Index (Nov 1 = 0)')
ax.set_title(
    f'Irrigation Season Length per Year\n'
    f'(AETI ≥ {IRRIGATION_THRESHOLD_MM} mm threshold)\n'
    f'Note: 2021/22 season start not fully captured — use with caution',
    fontweight='bold', fontsize=11
)
ax.grid(axis='x', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'growing_season_length.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 15 · Analysis Outputs — Data Export

All results are exported as structured CSV files for archiving, reproducibility, and downstream use. Inactive pivot-seasons have CWP masked to `NaN` prior to export, ensuring that bare-soil evaporation does not contaminate water productivity summaries in wide-form or aggregated tables.

| File | Rows | Content |
|---|---|---|
| `seasonal_performance_summary.csv` | 4 | Project-level seasonal aggregates |
| `pivot_operational_classification.csv` | 166 | Per-pivot class, WPb, trend slope, centroid coordinates |
| `master_seasonal_statistics.csv` | 664 | Long-form: one row per pivot × season |
| `master_dekadal_phenology.csv` | ~5,742 | Dekadal AETI and NPP for active pivots |
| `pivot_cwp_wide_trend.csv` | 166 | Wide-form CWP (inactive seasons = NaN) |
| `season_windows.csv` | 4 | Irrigation season start, end, length, peak dekad |

In [ ]:
# --- 0. Data Cleaning: Ensure Masked CWP for Inactive Seasons ---
# This ensures bare-soil evaporation from inactive seasons doesn't appear in the WP trend files
stats_df.loc[stats_df['is_active'] == 0, 'cwp'] = np.nan

# --- 1. Seasonal Performance Summary (FINAL VERIFIED LOGIC) ---
# Aggregating only active pivots ensures the summary table and CSV are correct.
seasonal_summary = stats_df[stats_df['is_active'] == 1].groupby('season').agg({
    'pivot_id': 'count',
    'area_ha': 'sum',
    'cwp': ['mean', 'median'],
    'abstraction_vol_m3': 'sum'
}).reset_index()

seasonal_summary.columns = ['Season', 'Active_Pivots', 'Total_Area_Ha', 'Mean_WP', 'Median_WP', 'Demand_Volume_m3']
seasonal_summary['Total_Volume_MCM'] = seasonal_summary['Demand_Volume_m3'] / 1e6

seasonal_summary.to_csv(os.path.join(tables_dir, 'seasonal_performance_summary.csv'), index=False)

# --- 2. Pivot Operational Classification (with GPS Centroids) ---
pivots_export = pivots_gdf.copy()
centroids_latlon = pivots_export.geometry.centroid.to_crs('EPSG:4326')
pivots_export['lon'] = centroids_latlon.x
pivots_export['lat'] = centroids_latlon.y

classification_cols = ['pivot_id', 'area_ha', 'n_active_seasons', 'operation_class', 'avg_wpb_kg_m3', 'trend', 'p_value', 'slope', 'lat', 'lon']
pivots_export[classification_cols].to_csv(os.path.join(tables_dir, 'pivot_operational_classification.csv'), index=False)

# --- 3. Master Long-form Statistics (Seasonal) ---
# Now contains NaNs for CWP and 0.0 for consumption where is_active == 0
stats_df.to_csv(os.path.join(tables_dir, 'master_seasonal_statistics.csv'), index=False)

# --- 4. Master Dekadal Phenology (Time Series) ---
df_dek.to_csv(os.path.join(tables_dir, 'master_dekadal_phenology.csv'), index=False)

# --- 5. Seasonal Phenology Windows ---
windows_df.to_csv(os.path.join(tables_dir, 'season_windows.csv'), index=False)

# --- 6. Wide-form CWP Trend Table (Heatmap Backbone) ---
# The wide table now inherits the corrected NaN values for inactive seasons
pivot_cwp_wide = stats_df.pivot(index='pivot_id', columns='season', values='cwp')
pivot_cwp_wide.to_csv(os.path.join(tables_dir, 'pivot_cwp_wide_trend.csv'))

print(f'Sync Complete: All deliverables exported to: {tables_dir}')
print(f"Verified 2024/25 Peak Demand in CSV: {seasonal_summary.iloc[-1]['Total_Volume_MCM']:.2f} MCM")